<a href="https://colab.research.google.com/github/beyzadurdu6619/TrustLLM-Uncertainty-Quantification/blob/main/notebooks/07_week/semantic_uncertainty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Set seed for reproducibility / Tekrarlanabilirlik için tohum belirleme
torch.manual_seed(42)

# 2. Load model and tokenizer / Modeli ve tokenizer'ı yükleme
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 3. Define prompt categories for semantic uncertainty evaluation
# Anlamsal belirsizlik değerlendirmesi için soru kategorilerini tanımlama
prompts = {
    # Net/Certain Questions (Model should be certain)
    "certain": "What is the capital of France?",
    # Ambiguous Questions (Model might give subjective/varied answers)
    "ambiguous": "What is the best programming language for beginners?",
    # Unknown/Future Questions (Model is likely to hallucinate or be highly uncertain)
    "unknown": "What will be the world population in the year 2150?",
}

print("Model and Prompts Ready!")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model and Prompts Ready!


In [2]:
def generate_sampled_responses(
    model, tokenizer, prompt, num_samples=10, max_new_tokens=20
):
    """Generates N different responses for a given prompt using temperature sampling.

    TR: Verilen prompt için sıcaklık ölçeklemesiyle N adet farklı yanıt
    üretir.
    """
    inputs = tokenizer(prompt, return_tensors="pt")
    responses = []

    for _ in range(num_samples):
        # Enable sampling with temperature T=0.7
        # T=0.7 esnekliği ile sampling işlemini aktif ediyoruz
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

        # Decode only the generated response text (excluding the prompt)
        # Sadece modelin ürettiği yeni metni çözümlüyoruz (prompt hariç)
        generated_text = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1] :], skip_special_tokens=True
        )
        responses.append(generated_text.strip())

    return responses


# Test generation on a sample prompt / Örnek soru üzerinde testi çalıştıralım
sample_responses = generate_sampled_responses(
    model, tokenizer, prompts["certain"], num_samples=5
)
print("Sample Responses:", sample_responses)

Sample Responses: ['The capital of France is the capital of France, and it is the capital of France.', 'The capital of France is Leuven. The capital of France is Paris.', 'Paris is a city of 500,000 people, and we are one of the most important', 'The capital of France is Paris. It is a capital of France. The capital of France', '"We have a capital of more than $8 billion, and we have an investment of']


In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

# Load lightweight embedding model / Hafif embedding modelini yüklüyoruz
embedder = SentenceTransformer("all-MiniLM-L6-v2")


def cluster_responses_by_meaning(responses, distance_threshold=0.3):
    """Clusters generated responses based on semantic similarity using hierarchical clustering.

    TR: Üretilen metin yanıtlarını anlamsal benzerliklerine göre gruplar.
    """
    # Convert text responses into embedding vectors
    # Metin yanıtlarını vektörlere dönüştürüyoruz
    embeddings = embedder.encode(responses)

    # Agglomerative clustering based on cosine distance
    # Kosinüs mesafesine göre hiyerarşik kümeleme
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="cosine",
        linkage="average",
        distance_threshold=distance_threshold,
    )

    cluster_labels = clustering.fit_predict(embeddings)
    return cluster_labels


# Test clustering / Kümeleme fonksiyonunu test edelim
cluster_labels = cluster_responses_by_meaning(sample_responses)
print("Assigned Cluster Labels:", cluster_labels)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Assigned Cluster Labels: [0 0 2 0 1]


In [4]:
import numpy as np


def compute_semantic_entropy(cluster_labels):
    """Computes Semantic Entropy over the probability distribution of response clusters.

    TR: Anlamsal kümelerin olasılık dağılımı üzerinden Entropi (Belirsizlik)
    hesaplar.
    """
    # Count occurrences of each cluster / Her kümedeki yanıt sayısını buluyoruz
    _, counts = np.unique(cluster_labels, return_counts=True)

    # Calculate empirical probability distribution / Olasılık dağılımını hesaplıyoruz
    probabilities = counts / len(cluster_labels)

    # Entropy formula / Entropi formülü
    semantic_entropy = -np.sum(probabilities * np.log(probabilities + 1e-12))
    return semantic_entropy


# Compare uncertainty across all 3 prompt categories
# 3 farklı soru kategorisindeki belirsizlik skorlarını karşılaştıralım
results = {}
for category, prompt_text in prompts.items():
    resps = generate_sampled_responses(
        model, tokenizer, prompt_text, num_samples=10
    )
    c_labels = cluster_responses_by_meaning(resps)
    entropy_score = compute_semantic_entropy(c_labels)
    results[category] = entropy_score
    print(f"[{category.upper()}] Semantic Entropy: {entropy_score:.4f}")

[CERTAIN] Semantic Entropy: 1.7481
[AMBIGUOUS] Semantic Entropy: 1.4185
[UNKNOWN] Semantic Entropy: 1.9730
